In [ ]:
# Ячейка 1: установка (запустить один раз)
# import Pkg; Pkg.add("Interact"); Pkg.add("WebIO")

# ── Ячейка 2: всё остальное ──────────────────────────────────────────────────

using Plots, Interact, Random
gr()

CITIES_I = [
    565.0 575.0;  25.0 185.0; 345.0 750.0; 945.0 685.0; 845.0 655.0;
    880.0 660.0;  25.0 230.0; 525.0 1000.0; 580.0 1175.0; 650.0 1130.0;
    1605.0 620.0; 1220.0 580.0; 1465.0 200.0; 1530.0 5.0; 845.0 680.0;
    725.0 370.0; 145.0 665.0; 415.0 635.0; 510.0 875.0; 560.0 365.0;
    300.0 465.0; 520.0 585.0; 480.0 415.0; 835.0 625.0; 975.0 580.0;
    1215.0 245.0; 1320.0 315.0; 1250.0 400.0; 660.0 180.0; 410.0 250.0;
    420.0 555.0; 575.0 665.0; 1150.0 1160.0; 700.0 580.0; 685.0 595.0;
    685.0 610.0; 770.0 610.0; 795.0 645.0; 720.0 635.0; 760.0 650.0;
    475.0 960.0; 95.0 260.0; 875.0 920.0; 700.0 500.0; 555.0 815.0;
    830.0 485.0; 1170.0 65.0; 830.0 610.0; 605.0 625.0; 595.0 360.0;
    1340.0 725.0; 1740.0 245.0
]
OPT_DIST_I = 7542
N_I = size(CITIES_I, 1)

function _dist_mat(c)
    n = size(c,1); D = zeros(Int,n,n)
    for i in 1:n, j in 1:n
        D[i,j] = round(Int, sqrt((c[i,1]-c[j,1])^2 + (c[i,2]-c[j,2])^2))
    end; D
end
DIST_I = _dist_mat(CITIES_I)

function _tlen(tour, D)
    s=0; n=length(tour)
    for i in 1:n; s+=D[tour[i],tour[(i%n)+1]]; end; s
end

function _rsel(w)
    r=rand()*sum(w); cs=0.0
    for i in eachindex(w); cs+=w[i]; cs>=r && return i; end
    length(w)
end

function _btour(τ,α,β,D;q0=0.0)
    n=size(τ,1); st=rand(1:n); vis=falses(n); vis[st]=true
    tour=Vector{Int}(undef,n); tour[1]=st; cd=Int[]; ws=Float64[]
    for step in 2:n
        cur=tour[step-1]; empty!(cd); empty!(ws)
        for j in 1:n; !vis[j]||continue; push!(cd,j); push!(ws,(τ[cur,j]^α)*((1.0/D[cur,j])^β)); end
        nx=(q0>0&&rand()<q0) ? cd[argmax(ws)] : cd[_rsel(ws)]
        tour[step]=nx; vis[nx]=true
    end; tour
end

function _run(;n_ants=52,α=1.0,β=5.0,ρ=0.5,Q=100.0,τ0=1e-6,n_gen=80,
               elitism=0,M_dep=0,τ_min=-1.0,τ_max=-1.0,ϕ=0.0,q0=0.0,seed=42)
    Random.seed!(seed); n=N_I; D=DIST_I; τ=fill(τ0,n,n)
    gbt=collect(1:n); gbd=typemax(Int)
    at=Vector{Int}[]; ad=Int[]; hb=Int[]; ha=Float64[]
    for gen in 1:n_gen
        tours=[_btour(τ,α,β,D;q0=q0) for _ in 1:n_ants]
        if ϕ>0; for t in tours,s in 1:n; a,b=t[s],t[(s%n)+1]
            τ[a,b]=(1-ϕ)*τ[a,b]+ϕ*τ0; τ[b,a]=(1-ϕ)*τ[b,a]+ϕ*τ0; end; end
        ls=[_tlen(t,D) for t in tours]; ord=sortperm(ls)
        if ls[ord[1]]<gbd; gbd=ls[ord[1]]; gbt=copy(tours[ord[1]]); end
        push!(at,copy(gbt)); push!(ad,gbd); push!(hb,gbd); push!(ha,sum(ls)/n_ants)
        τ.*=(1-ρ)
        deps=M_dep>0 ? ord[1:min(M_dep,n_ants)] : collect(1:n_ants)
        for k in deps; Δ=Q/ls[k]; t=tours[k]
            for s in 1:n; a,b=t[s],t[(s%n)+1]; τ[a,b]+=Δ; τ[b,a]+=Δ; end; end
        if elitism>0; for e in 1:min(elitism,n_ants); k=ord[e]; Δ=Q/ls[k]; t=tours[k]
            for s in 1:n; a,b=t[s],t[(s%n)+1]; τ[a,b]+=Δ; τ[b,a]+=Δ; end; end; end
        if τ_min>=0; clamp!(τ,τ_min,τ_max); end
    end
    at,ad,hb,ha
end

# ── Прогон 4 алгоритмов ─────────────────────────────────────────────────────

println("  Прогон...")
cfgs = [
    ("AS",         :dodgerblue,   Dict()),
    ("Elitist AS", :orangered,    Dict(:elitism=>2)),
    ("MMAS",       :seagreen,     Dict(:M_dep=>4, :τ_min=>1e-8, :τ_max=>10.0)),
    ("ACS",        :mediumpurple, Dict(:M_dep=>1, :ϕ=>0.1, :q0=>0.9)),
]

DATA = Dict{String,Any}()
for (nm,col,kw) in cfgs
    tours,dists,hb,ha = _run(;n_gen=80, kw...)
    DATA[nm] = (tours=tours, dists=dists, hb=hb, ha=ha, col=col)
    println("  ▸ $nm → $(dists[end])")
end

algo_names = [c[1] for c in cfgs]
algo_cols  = [c[2] for c in cfgs]
n_gen = 80

# ── Интерактивный виджет ─────────────────────────────────────────────────────

ui_gen  = slider(1:n_gen, value=n_gen, label="Поколение")
ui_algo = dropdown(algo_names, value=algo_names[1], label="Алгоритм")

map(ui_gen, ui_algo) do gen, alg
    d = DATA[alg]
    g = clamp(gen, 1, length(d.tours))
    tour = d.tours[g]
    xs = [CITIES_I[tour,1]; CITIES_I[tour[1],1]]
    ys = [CITIES_I[tour,2]; CITIES_I[tour[1],2]]
    gap = round((d.dists[g] - OPT_DIST_I) / OPT_DIST_I * 100; digits=1)

    l = @layout [a{0.55w} [b; c]]

    # Маршрут
    p1 = plot(xs, ys; linewidth=1.8, linecolor=d.col, linealpha=0.85,
        label=nothing, title="$alg · пок. $g · L=$(d.dists[g]) (+$(gap)%)",
        titlefontsize=10, aspect_ratio=:equal, grid=false, framestyle=:box,
        xticks=[], yticks=[], xlims=(-50,1850), ylims=(-100,1300))
    scatter!(p1, CITIES_I[:,1], CITIES_I[:,2];
        markersize=3, markercolor=d.col, markeralpha=0.7, markerstrokewidth=0, label=nothing)

    # Схождение
    p2 = plot(; xlabel="Поколение", ylabel="Лучшее расстояние",
        title="Схождение", titlefontsize=9,
        xlims=(0,n_gen+1), ylims=(OPT_DIST_I*0.95, maximum(d.hb)*1.05),
        grid=true, gridstyle=:dot, gridalpha=0.3)
    for (nm2,col2,_) in cfgs
        d2 = DATA[nm2]
        plot!(p2, 1:n_gen, d2.hb; linewidth=1.2, linecolor=d2.col,
            linealpha=(nm2==alg ? 1.0 : 0.3), label=nm2)
    end
    hline!(p2, [OPT_DIST_I]; label="Опт.", linecolor=:red, linestyle=:dash, linewidth=1.5)
    vline!(p2, [g]; label=nothing, linecolor=:gray40, linestyle=:dot, linewidth=1)

    # Столбцы
    cur_dists = [DATA[nm].dists[clamp(g,1,length(DATA[nm].dists))] for nm in algo_names]
    cur_gaps  = [round((cd - OPT_DIST_I)/OPT_DIST_I*100; digits=1) for cd in cur_dists]
    p3 = bar(algo_names, cur_dists; color=algo_cols, label=nothing, bar_width=0.6,
        ylabel="Расстояние", title="Сравнение (пок. $g)", titlefontsize=9,
        ylims=(OPT_DIST_I*0.93, maximum(cur_dists)*1.07))
    hline!(p3, [OPT_DIST_I]; label=nothing, linecolor=:red, linestyle=:dash, linewidth=1.5)
    for (i,(cd,cg)) in enumerate(zip(cur_dists, cur_gaps))
        annotate!(p3, i, cd+120, Plots.text("+$(cg)%", 7, :center))
    end

    plot(p1, p2, p3; layout=l, size=(1100, 500), margin=4Plots.mm)
end

  Прогон 4 алгоритмов...
  ▸ AS... → 7679
  ▸ Elitist AS... → 7679
  ▸ MMAS... → 7558
  ▸ ACS... → 8460

  Дашборд открыт. Двигай слайдер и выбирай алгоритм в меню!



In [1]:
import Pkg
Pkg.add(["Interact", "WebIO"])

   Resolving package versions...
   Installed DelaunayTriangulation ─ v1.6.6


LoadError: Unable to automatically download/install artifact 'OpenEXR' from sources listed in 'C:\Users\Admin\.julia\packages\OpenEXR_jll\SDJvz\Artifacts.toml'.
Sources attempted:
- https://pkg.julialang.org/artifact/6ae65427b56f466b2f1b215fffaa8935b2ac3e65
    Error: RequestError: schannel: failed to receive handshake, SSL/TLS connection failed while requesting https://pkg.julialang.org/artifact/6ae65427b56f466b2f1b215fffaa8935b2ac3e65
- https://github.com/JuliaBinaryWrappers/OpenEXR_jll.jl/releases/download/OpenEXR-v3.4.4+0/OpenEXR.v3.4.4.x86_64-w64-mingw32-cxx11.tar.gz
    Error: RequestError: schannel: failed to receive handshake, SSL/TLS connection failed while requesting https://github.com/JuliaBinaryWrappers/OpenEXR_jll.jl/releases/download/OpenEXR-v3.4.4+0/OpenEXR.v3.4.4.x86_64-w64-mingw32-cxx11.tar.gz


In [ ]:
using Plots, Random
gr()

CITIES_S = [
    565.0 575.0;  25.0 185.0; 345.0 750.0; 945.0 685.0; 845.0 655.0;
    880.0 660.0;  25.0 230.0; 525.0 1000.0; 580.0 1175.0; 650.0 1130.0;
    1605.0 620.0; 1220.0 580.0; 1465.0 200.0; 1530.0 5.0; 845.0 680.0;
    725.0 370.0; 145.0 665.0; 415.0 635.0; 510.0 875.0; 560.0 365.0;
    300.0 465.0; 520.0 585.0; 480.0 415.0; 835.0 625.0; 975.0 580.0;
    1215.0 245.0; 1320.0 315.0; 1250.0 400.0; 660.0 180.0; 410.0 250.0;
    420.0 555.0; 575.0 665.0; 1150.0 1160.0; 700.0 580.0; 685.0 595.0;
    685.0 610.0; 770.0 610.0; 795.0 645.0; 720.0 635.0; 760.0 650.0;
    475.0 960.0; 95.0 260.0; 875.0 920.0; 700.0 500.0; 555.0 815.0;
    830.0 485.0; 1170.0 65.0; 830.0 610.0; 605.0 625.0; 595.0 360.0;
    1340.0 725.0; 1740.0 245.0
]
OPT_S = 7542
N_S = size(CITIES_S, 1)

function _dm(c)
    n=size(c,1); D=zeros(Int,n,n)
    for i in 1:n, j in 1:n
        D[i,j]=round(Int,sqrt((c[i,1]-c[j,1])^2+(c[i,2]-c[j,2])^2))
    end; D
end
DS = _dm(CITIES_S)

function _tl(tour,D)
    s=0; n=length(tour)
    for i in 1:n; s+=D[tour[i],tour[(i%n)+1]]; end; s
end

function _rs(w)
    r=rand()*sum(w); cs=0.0
    for i in eachindex(w); cs+=w[i]; cs>=r && return i; end; length(w)
end

function _bt(τ,α,β,D;q0=0.0)
    n=size(τ,1); st=rand(1:n); vis=falses(n); vis[st]=true
    tour=Vector{Int}(undef,n); tour[1]=st; cd=Int[]; ws=Float64[]
    for step in 2:n
        cur=tour[step-1]; empty!(cd); empty!(ws)
        for j in 1:n; !vis[j]||continue; push!(cd,j)
            push!(ws,(τ[cur,j]^α)*((1.0/D[cur,j])^β)); end
        nx=(q0>0&&rand()<q0) ? cd[argmax(ws)] : cd[_rs(ws)]
        tour[step]=nx; vis[nx]=true
    end; tour
end

# Универсальный ACO — возвращает history_best
function run_study(;n_ants=52,α=1.0,β=5.0,ρ=0.5,Q=100.0,τ0=1e-6,n_gen=80,
                    elitism=0,M_dep=0,τ_min=-1.0,τ_max=-1.0,ϕ=0.0,q0=0.0,seed=-1)
    seed>=0 && Random.seed!(seed)
    n=N_S; D=DS; τ=fill(τ0,n,n)
    gbd=typemax(Int); hb=Int[]
    for gen in 1:n_gen
        tours=[_bt(τ,α,β,D;q0=q0) for _ in 1:n_ants]
        if ϕ>0; for t in tours,s in 1:n; a,b=t[s],t[(s%n)+1]
            τ[a,b]=(1-ϕ)*τ[a,b]+ϕ*τ0; τ[b,a]=(1-ϕ)*τ[b,a]+ϕ*τ0; end; end
        ls=[_tl(t,D) for t in tours]; ord=sortperm(ls)
        if ls[ord[1]]<gbd; gbd=ls[ord[1]]; end
        push!(hb,gbd)
        τ.*=(1-ρ)
        deps=M_dep>0 ? ord[1:min(M_dep,n_ants)] : collect(1:n_ants)
        for k in deps; Δ=Q/ls[k]; t=tours[k]
            for s in 1:n; a,b=t[s],t[(s%n)+1]; τ[a,b]+=Δ; τ[b,a]+=Δ; end; end
        if elitism>0; for e in 1:min(elitism,n_ants); k=ord[e]; Δ=Q/ls[k]; t=tours[k]
            for s in 1:n; a,b=t[s],t[(s%n)+1]; τ[a,b]+=Δ; τ[b,a]+=Δ; end; end; end
        if τ_min>=0; clamp!(τ,τ_min,τ_max); end
    end
    hb
end

# Усреднение по MC-прогонам
function mc_run(n_mc; kwargs...)
    all_hb = [run_study(; seed=s, kwargs...) for s in 1:n_mc]
    n_gen = length(all_hb[1])
    mean_hb = [sum(all_hb[r][g] for r in 1:n_mc)/n_mc for g in 1:n_gen]
    return mean_hb
end

mkpath("lab16")
N_MC = 10
N_GEN = 80

# ═══════════════════════════════════════════════════════════════════════════════
# 1. Зависимость от числа муравьёв N (на примере MMAS)
# ═══════════════════════════════════════════════════════════════════════════════

println("  1/4  Исследование N (число муравьёв)...")

ant_counts = [10, 26, 52, 104]
ant_colors = [:royalblue, :seagreen, :darkorange, :crimson]

p_ants = plot(; xlabel="Поколение", ylabel="Лучшее расстояние (ср. по $N_MC MC)",
    title="MMAS: зависимость сходимости от числа муравьёв N",
    titlefontsize=10, legend=:topright, grid=true, gridstyle=:dot, gridalpha=0.3,
    size=(800,450))

for (i, na) in enumerate(ant_counts)
    hb = mc_run(N_MC; n_ants=na, n_gen=N_GEN, M_dep=4, τ_min=1e-8, τ_max=10.0)
    plot!(p_ants, 1:N_GEN, hb; linewidth=2, linecolor=ant_colors[i],
        label="N=$na (→$(round(Int,hb[end])))")
end
hline!(p_ants, [OPT_S]; linecolor=:red, linestyle=:dash, linewidth=1.5, label="Опт. ($OPT_S)")
display(p_ants)
savefig(p_ants, "lab16/study_n_ants.png")
println("    ✓ lab16/study_n_ants.png")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. Зависимость от ρ (скорость испарения)
# ═══════════════════════════════════════════════════════════════════════════════

println("  2/4  Исследование ρ (испарение)...")

rho_vals = [0.1, 0.3, 0.5, 0.9]
rho_colors = [:royalblue, :seagreen, :darkorange, :crimson]

p_rho = plot(; xlabel="Поколение", ylabel="Лучшее расстояние (ср. по $N_MC MC)",
    title="MMAS: зависимость сходимости от скорости испарения ρ",
    titlefontsize=10, legend=:topright, grid=true, gridstyle=:dot, gridalpha=0.3,
    size=(800,450))

for (i, rv) in enumerate(rho_vals)
    hb = mc_run(N_MC; n_ants=52, ρ=rv, n_gen=N_GEN, M_dep=4, τ_min=1e-8, τ_max=10.0)
    plot!(p_rho, 1:N_GEN, hb; linewidth=2, linecolor=rho_colors[i],
        label="ρ=$rv (→$(round(Int,hb[end])))")
end
hline!(p_rho, [OPT_S]; linecolor=:red, linestyle=:dash, linewidth=1.5, label="Опт. ($OPT_S)")
display(p_rho)
savefig(p_rho, "lab16/study_rho.png")
println("    ✓ lab16/study_rho.png")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. Зависимость от β (вес эвристики)
# ═══════════════════════════════════════════════════════════════════════════════

println("  3/4  Исследование β (вес эвристики)...")

beta_vals = [1.0, 3.0, 5.0, 10.0]
beta_colors = [:royalblue, :seagreen, :darkorange, :crimson]

p_beta = plot(; xlabel="Поколение", ylabel="Лучшее расстояние (ср. по $N_MC MC)",
    title="AS: зависимость сходимости от β (вес расстояния)",
    titlefontsize=10, legend=:topright, grid=true, gridstyle=:dot, gridalpha=0.3,
    size=(800,450))

for (i, bv) in enumerate(beta_vals)
    hb = mc_run(N_MC; n_ants=52, β=bv, n_gen=N_GEN)
    plot!(p_beta, 1:N_GEN, hb; linewidth=2, linecolor=beta_colors[i],
        label="β=$bv (→$(round(Int,hb[end])))")
end
hline!(p_beta, [OPT_S]; linecolor=:red, linestyle=:dash, linewidth=1.5, label="Опт. ($OPT_S)")
display(p_beta)
savefig(p_beta, "lab16/study_beta.png")
println("    ✓ lab16/study_beta.png")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. Сравнение скорости сходимости 4 алгоритмов (MC-усреднение)
# ═══════════════════════════════════════════════════════════════════════════════

println("  4/4  Сравнение 4 алгоритмов (MC=$N_MC)...")

algo_cfgs = [
    ("AS",         :dodgerblue,   Dict(:n_ants=>52)),
    ("Elitist AS", :orangered,    Dict(:n_ants=>52, :elitism=>2)),
    ("MMAS",       :seagreen,     Dict(:n_ants=>52, :M_dep=>4, :τ_min=>1e-8, :τ_max=>10.0)),
    ("ACS",        :mediumpurple, Dict(:n_ants=>52, :M_dep=>1, :ϕ=>0.1, :q0=>0.9)),
]

p_cmp = plot(; xlabel="Поколение", ylabel="Лучшее расстояние (ср. по $N_MC MC)",
    title="Сравнение скорости сходимости 4 алгоритмов ACO",
    titlefontsize=10, legend=:topright, grid=true, gridstyle=:dot, gridalpha=0.3,
    size=(800,450))

final_results = Dict{String,Float64}()
for (nm, col, kw) in algo_cfgs
    hb = mc_run(N_MC; n_gen=N_GEN, kw...)
    final_results[nm] = hb[end]
    plot!(p_cmp, 1:N_GEN, hb; linewidth=2, linecolor=col,
        label="$nm (→$(round(Int,hb[end])))")
end
hline!(p_cmp, [OPT_S]; linecolor=:red, linestyle=:dash, linewidth=1.5, label="Опт. ($OPT_S)")
display(p_cmp)
savefig(p_cmp, "lab16/study_algo_comparison.png")
println("    ✓ lab16/study_algo_comparison.png")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. Финальная сводка: столбцы + N-зависимость в одном дашборде
# ═══════════════════════════════════════════════════════════════════════════════

names_s = [c[1] for c in algo_cfgs]
vals_s  = [final_results[n] for n in names_s]
cols_s  = [c[2] for c in algo_cfgs]
gaps_s  = [(v - OPT_S)/OPT_S*100 for v in vals_s]

p_bar = bar(names_s, vals_s; color=cols_s, label=nothing, bar_width=0.6,
    ylabel="Среднее лучшее (MC=$N_MC)", title="Финальное качество после $N_GEN поколений",
    titlefontsize=10, ylims=(OPT_S*0.93, maximum(vals_s)*1.07), size=(600,400))
hline!(p_bar, [OPT_S]; linecolor=:red, linestyle=:dash, linewidth=1.5, label="Оптимум")
for (i,(v,g)) in enumerate(zip(vals_s, gaps_s))
    annotate!(p_bar, i, v+100, Plots.text("+$(round(g;digits=1))%", 8, :center))
end
display(p_bar)
savefig(p_bar, "lab16/study_final_bar.png")
println("    ✓ lab16/study_final_bar.png")

p_dash = plot(p_cmp, p_ants, p_rho, p_beta;
    layout=(2,2), size=(1200,900),
    plot_title="Исследование параметров ACO — Berlin52",
    plot_titlefontsize=12, margin=5Plots.mm)
display(p_dash)
savefig(p_dash, "lab16/study_dashboard.png")
println("    ✓ lab16/study_dashboard.png")

println("\n  Все графики исследования сохранены в lab16/\n")